In [1]:
#!/usr/bin/env python
# coding: utf-8

# # Simple Load CSV Files Template Test
# 
# This notebook directly tests the load_csv_files_with_config template without any Galaxy or wrapper complexity.
# Similar to the Ripley example that uses: `run_ripley(ripley_params, save_results=False)`

# ## Step 1: Setup Environment

import os
import json
import pandas as pd
import sys
from pathlib import Path

# Create working directory
work_dir = Path("load_csv_test")
work_dir.mkdir(exist_ok=True)
os.chdir(work_dir)
print(f"Working in: {os.getcwd()}")

# ## Step 2: Create Test CSV Files

# Create test CSV file 1
data1 = pd.DataFrame({
    'CellID': [1, 2, 3, 4, 5],
    'CD45': [10.5, 20.3, 15.7, 8.2, 12.4],
    'CD3D': [5.2, 8.1, 3.4, 12.3, 6.7],
    'X_centroid': [100, 150, 200, 250, 300],
    'Y_centroid': [100, 120, 140, 160, 180],
    'cell_type': ['T cell', 'B cell', 'T cell', 'Macrophage', 'B cell']
})

# Create test CSV file 2  
data2 = pd.DataFrame({
    'CellID': [6, 7, 8, 9, 10],
    'CD45': [11.5, 22.3, 13.7, 9.2, 14.4],
    'CD3D': [6.2, 7.1, 4.4, 11.3, 5.7],
    'X_centroid': [350, 400, 450, 500, 550],
    'Y_centroid': [200, 220, 240, 260, 280],
    'cell_type': ['T cell', 'NK cell', 'B cell', 'Macrophage', 'T cell']
})

# Save CSV files
csv_dir = Path("csv_files")
csv_dir.mkdir(exist_ok=True)

data1.to_csv(csv_dir / "sample_data_1.csv", index=False)
data2.to_csv(csv_dir / "sample_data_2.csv", index=False)
print(f"Created CSV files in {csv_dir}:")
print(f"  - sample_data_1.csv ({data1.shape})")
print(f"  - sample_data_2.csv ({data2.shape})")

# ## Step 3: Create Configuration Table

config_df = pd.DataFrame({
    'file_name': ['sample_data_1.csv', 'sample_data_2.csv'],
    'slide_number': ['Slide_001', 'Slide_002'],
    'patient_id': ['Patient_A', 'Patient_A'],
    'timepoint': ['baseline', 'week4']
})

config_df.to_csv('config_table.csv', index=False)
print("\nConfiguration table created:")
print(config_df)

# ## Step 4: Create Parameters JSON

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Parameters matching the template's expected format
params = {
    "CSV_Files": str(csv_dir.absolute()),  # Directory containing CSV files
    "CSV_Files_Configuration": "config_table.csv",  # Configuration file
    "Output_File": str(output_dir / "combined_data.csv"),  # Where to save output
    "String_Columns": ["cell_type", "slide_number", "patient_id", "timepoint"]  # Columns to treat as strings
}

# Save parameters to JSON file
with open('load_csv_params.json', 'w') as f:
    json.dump(params, f, indent=2)

print("\nParameters JSON created:")
print(json.dumps(params, indent=2))

# ## Step 5: Import and Run the Template
# 
# **IMPORTANT:** You need to copy your `load_csv_files_with_config_template.py` 
# file to the current directory first!

template_file = Path("../../../src/load_csv_files_with_config_template.py")
if template_file.exists():
    import shutil
    shutil.copy(template_file, "load_csv_files_with_config_template.py")
    print(f"\n✓ Template copied from {template_file}")
else:
    print(f"\n⚠️  Please copy the template file to: {Path.cwd() / 'load_csv_files_with_config_template.py'}")
    print("Then continue with the next cell...")

# ## Step 6: Execute the Template

# Import the template module
import importlib.util

template_path = "load_csv_files_with_config_template.py"
if not Path(template_path).exists():
    print(f"❌ Template file not found: {template_path}")
    print("Please copy the template file and try again.")
else:
    # Load the template module
    spec = importlib.util.spec_from_file_location("load_csv_template", template_path)
    template_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(template_module)
    
    print("✓ Template module loaded successfully")
    print(f"Available functions: {[x for x in dir(template_module) if not x.startswith('_')]}")
    
    # Check if run_from_json exists
    if hasattr(template_module, 'run_from_json'):
        print("\n✓ Found run_from_json function")
        
        # Run the template with the parameters JSON
        print("\n" + "="*50)
        print("EXECUTING TEMPLATE...")
        print("="*50)
        
        try:
            # Execute the template (similar to how Ripley example runs)
            result = template_module.run_from_json(
                'load_csv_params.json',
                save_results=True,  # Save output files
                show_plot=False     # No plots for this template
            )
            
            print("\n✓ Template executed successfully!")
            if result is not None:
                print(f"  Result type: {type(result).__name__}")
                if isinstance(result, pd.DataFrame):
                    print(f"  Result shape: {result.shape}")
                    
        except Exception as e:
            print(f"\n❌ Error during execution: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("\n❌ run_from_json function not found in template")
        
        # Try alternative approach - direct function call
        if hasattr(template_module, 'LoadCSVFilesWithConfig'):
            print("\n  Found LoadCSVFilesWithConfig class - trying direct instantiation...")
            # You might need to adjust this based on the actual class structure

# ## Step 7: Check the Output

output_file = output_dir / "combined_data.csv"
if output_file.exists():
    print("\n" + "="*50)
    print("OUTPUT VERIFICATION")
    print("="*50)
    
    # Load and inspect the output
    combined_df = pd.read_csv(output_file)
    print(f"\n✓ Output file created: {output_file}")
    print(f"  Shape: {combined_df.shape}")
    print(f"  Columns: {list(combined_df.columns)}")
    
    # Check if metadata was added
    if 'slide_number' in combined_df.columns:
        print("\n✓ Metadata columns successfully added:")
        print(f"  - slide_number values: {combined_df['slide_number'].unique()}")
        print(f"  - patient_id values: {combined_df['patient_id'].unique()}")
        print(f"  - timepoint values: {combined_df['timepoint'].unique()}")
    
    # Show first few rows
    print("\nFirst 5 rows of combined data:")
    print(combined_df.head())
    
    # Show data types
    print("\nData types:")
    print(combined_df.dtypes)
    
    # Verify string columns
    string_cols = params["String_Columns"]
    print(f"\nString columns verification:")
    for col in string_cols:
        if col in combined_df.columns:
            dtype = combined_df[col].dtype
            is_string = dtype == 'object' or dtype == 'string'
            status = "✓" if is_string else "✗"
            print(f"  {status} {col}: {dtype}")
else:
    print(f"\n⚠️  Output file not found: {output_file}")
    print("\nChecking for any other output files...")
    
    # Check output directory
    if output_dir.exists():
        files = list(output_dir.glob("*"))
        if files:
            print(f"Files in output directory:")
            for f in files:
                print(f"  - {f.name}")
        else:
            print("Output directory is empty")
    
    # Check current directory for any CSV outputs
    csv_files = list(Path.cwd().glob("*.csv"))
    if csv_files:
        print(f"\nCSV files in current directory:")
        for f in csv_files:
            print(f"  - {f.name}")

# ## Step 8: Summary

print("\n" + "="*50)
print("TEST SUMMARY")
print("="*50)

print("\nThis simple test demonstrates:")
print("1. Creating test CSV files with sample data")
print("2. Creating a configuration table with metadata")
print("3. Setting up parameters JSON with proper paths")
print("4. Running the template using run_from_json()")
print("5. Verifying the output was created correctly")

print("\nKey differences from Galaxy integration:")
print("- No wrapper script needed")
print("- Direct Python execution")
print("- Simple file paths (no Galaxy dataset handling)")
print("- Parameters as clean Python dict/JSON")

print("\nIf this works, the template is functioning correctly,")
print("and any Galaxy issues are related to the integration layer.")

Working in: /Users/liuf9/Projects/SCSAWorkflow/examples/spac_load_csv/load_csv_test
Created CSV files in csv_files:
  - sample_data_1.csv ((5, 6))
  - sample_data_2.csv ((5, 6))

Configuration table created:
           file_name slide_number patient_id timepoint
0  sample_data_1.csv    Slide_001  Patient_A  baseline
1  sample_data_2.csv    Slide_002  Patient_A     week4

Parameters JSON created:
{
  "CSV_Files": "/Users/liuf9/Projects/SCSAWorkflow/examples/spac_load_csv/load_csv_test/csv_files",
  "CSV_Files_Configuration": "config_table.csv",
  "Output_File": "output/combined_data.csv",
  "String_Columns": [
    "cell_type",
    "slide_number",
    "patient_id",
    "timepoint"
  ]
}

⚠️  Please copy the template file to: /Users/liuf9/Projects/SCSAWorkflow/examples/spac_load_csv/load_csv_test/load_csv_files_with_config_template.py
Then continue with the next cell...
❌ Template file not found: load_csv_files_with_config_template.py
Please copy the template file and try again.

⚠️  Outp